
#### Desafio - Partidas + Estádios

In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
%run "../Modulo 4/Variaveis"

In [0]:
partidas_df = spark.read.parquet(f"{caminho_silver}partidas")

In [0]:
estadios_df = spark.read.parquet(f"{caminho_silver}estadios")


In [0]:
print(f"Existem {partidas_df.count()} linhas na tabela partidas")
print(f"Existem {estadios_df.count()} linhas na tabela estadios")

In [0]:
display(partidas_df)

In [0]:
partidas_df_filtrado = partidas_df.filter("id_torneio is not null")

In [0]:
partidas_df_filtrado.display()

In [0]:
estadios_df.display()

In [0]:
partidas_estadios = partidas_df_filtrado.join(
    estadios_df,
    on=partidas_df.id_estadio == estadios_df.id_estadio,
    how="left"
)

In [0]:
partidas_estadios = partidas_estadios.select(
    partidas_df_filtrado['*'],
    estadios_df['nm_cidade'],
    estadios_df['nm_pais']
)

In [0]:
display(partidas_estadios)

In [0]:
from pyspark.sql.functions import concat, col, lit

In [0]:
partidas_estadios = partidas_estadios.withColumn(
    'nm_localizacao', concat(col('nm_cidade'), lit(', '), col('nm_pais'))
)

In [0]:
display(partidas_estadios)

In [0]:
from pyspark.sql.functions import concat_ws

In [0]:
partidas_estadios = partidas_estadios.withColumn(
    'nm_localizacao', concat_ws(', ', col('nm_cidade'), col('nm_pais'))
)

In [0]:
partidas_estadios.display()

In [0]:
nome_tabela = 'fato_partidas'
caminho_saida = f'{caminho_gold}{nome_tabela}'

In [0]:
print(f'Vou salvar os dados em {caminho_saida}')

In [0]:
partidas_estadios.write.parquet(f"{caminho_saida}", mode="overwrite")

In [0]:
%sql
DROP table futdata.ft_partidas

In [0]:
%sql
SELECT current_catalog(), current_schema()

In [0]:
%sql
SHOW TABLES IN futdata

In [0]:
partidas_estadios.write.mode('overwrite').saveAsTable(f'futdata.ft_partidas')